# TakeMeter — Local Notebook (VS Code)

Run each cell **top to bottom** with the `.venv` kernel (top-right kernel picker → pick the one in `./.venv`).

Every code line has a `#` comment explaining it. Order:
1. Setup + labels
2. Load your CSV
3. Split 70/15/15
4. Tokenize
5. Fine-tune DistilBERT
6. Evaluate the fine-tuned model
7. Groq zero-shot baseline (same test set)
8. Compare + save results

## 1. Setup + labels
Import the tools we need and define our 3 labels.

In [ ]:
import json                                          # read/write the results file
from pathlib import Path                             # handle file paths safely
import numpy as np                                   # math on arrays of numbers
import pandas as pd                                  # read the CSV as a table
import torch                                         # the deep-learning engine
from sklearn.model_selection import train_test_split # split data into train/test
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix  # scoring tools
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments  # the model + trainer
import matplotlib.pyplot as plt                      # draw the confusion matrix picture

ROOT = Path.cwd()                                    # the project folder (where this notebook lives)
LABEL_MAP = {"analysis": 0, "hot_take": 1, "reaction": 2}  # our 3 labels -> numbers the model uses
ID2LABEL = {v: k for k, v in LABEL_MAP.items()}      # reverse: number -> label name
LABELS = list(LABEL_MAP)                             # just the names: ['analysis','hot_take','reaction']
MODEL = "distilbert-base-uncased"                    # the pre-trained model we fine-tune
SEED = 42                                            # fixed random seed = same split every run

print("labels:", LABELS)                             # show the labels
print("device:", "mps" if torch.backends.mps.is_available() else "cpu")  # mps = your Mac's GPU

## 2. Load your CSV
You should see 195 rows and 65 of each label.

In [ ]:
df = pd.read_csv(ROOT / "data" / "takemeter_dataset.csv")  # read the CSV into a table
df = df[["text", "label"]].dropna()                  # keep only text+label columns, drop empty rows
df["y"] = df["label"].map(LABEL_MAP)                  # add a numeric column: label name -> 0/1/2
assert df["y"].notna().all(), "a label is not in LABEL_MAP!"  # safety: every label must be known

print("total rows:", len(df))                        # how many comments loaded
print(df["label"].value_counts())                    # how many of each label
df.head()                                            # preview the first 5 rows

## 3. Split 70 / 15 / 15 (stratified)
Stratified = each split keeps the 1/3-per-label balance. Train = learn from. Test = final exam the model never saw.

In [ ]:
# first split: 70% train, 30% leftover (stratify keeps label balance; seed makes it repeatable)
train_df, tmp = train_test_split(df, test_size=0.30, stratify=df["y"], random_state=SEED)
# second split: cut the 30% leftover in half -> 15% validation, 15% test
val_df, test_df = train_test_split(tmp, test_size=0.50, stratify=tmp["y"], random_state=SEED)

test_df.to_csv(ROOT / "data" / "test_split.csv", index=False)  # save the test set to disk
print(f"train={len(train_df)}  val={len(val_df)}  test={len(test_df)}")  # show split sizes
print("test label counts:\n", test_df["label"].value_counts())  # confirm test is balanced

## 4. Tokenize
Models can't read words — only numbers. The tokenizer turns each comment into a list of token IDs.

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL)            # load DistilBERT's matching tokenizer

def encode(texts):                                   # helper: turn a list of texts into token numbers
    return tok(list(texts), truncation=True, padding=True, max_length=256)  # cut/pad to 256 tokens

class DS(torch.utils.data.Dataset):                  # wrapper so the Trainer can read our data
    def __init__(self, enc, labels):                 # store the tokens + their labels
        self.enc, self.labels = enc, list(labels)
    def __len__(self):                               # how many examples
        return len(self.labels)
    def __getitem__(self, i):                        # give back one example (tokens + label) as tensors
        item = {k: torch.tensor(v[i]) for k, v in self.enc.items()}
        item["labels"] = torch.tensor(self.labels[i])
        return item

train_ds = DS(encode(train_df["text"]), train_df["y"])  # tokenize the training comments
test_ds  = DS(encode(test_df["text"]),  test_df["y"])   # tokenize the test comments
print("tokenized. example token ids:", train_ds[0]["input_ids"][:12].tolist())  # peek at one

## 5. Fine-tune DistilBERT
**Hyperparameter note:** the usual default (3 epochs, lr 2e-5) *underfits* this small 136-row train set — the loss barely drops and it never predicts `reaction`. We use **10 epochs + lr 5e-5** so it actually learns. Watch the **loss go down**. ~1-2 min.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(  # load DistilBERT with a fresh 3-class head
    MODEL, num_labels=len(LABELS), id2label=ID2LABEL, label2id=LABEL_MAP)

args = TrainingArguments(                            # all the training settings:
    output_dir=str(ROOT / "scripts" / "_trainer_out"),  # where temp files go
    num_train_epochs=10,                             # passes over the data (raised from 3 -> 10)
    per_device_train_batch_size=16,                  # comments per step
    learning_rate=5e-5,                              # how big each learning step is (raised from 2e-5)
    warmup_ratio=0.1,                                # ease into the learning rate at the start
    logging_steps=10,                                # print loss every 10 steps
    save_strategy="no",                              # don't save checkpoints (not needed)
    report_to="none",                                # no external logging
    seed=SEED)                                        # repeatable

trainer = Trainer(model=model, args=args, train_dataset=train_ds)  # bundle model + settings + data
trainer.train()                                      # <-- this actually trains. watch the loss drop.

## 6. Evaluate the fine-tuned model
Run the test set through the trained model and score it.

In [ ]:
pred = trainer.predict(test_ds)                      # run the 30 test comments through the model
probs = torch.softmax(torch.tensor(pred.predictions), dim=1).numpy()  # turn scores into probabilities
y_pred = probs.argmax(1)                             # the predicted label = highest probability
conf = probs.max(1)                                  # how confident (the winning probability)
y_true = test_df["y"].to_numpy()                     # the real labels

ft_acc = accuracy_score(y_true, y_pred)              # overall accuracy
ft_report = classification_report(y_true, y_pred, target_names=LABELS, output_dict=True, zero_division=0)  # per-class scores (as data)
cm = confusion_matrix(y_true, y_pred)                # the confusion matrix grid

print(f"FINE-TUNED ACCURACY: {ft_acc:.3f}\n")        # show accuracy
print(classification_report(y_true, y_pred, target_names=LABELS, zero_division=0))  # show the table

# save each prediction (text, true, pred, confidence) so we can analyze mistakes later
out = test_df.copy()                                 # copy the test table
out["pred"] = [ID2LABEL[i] for i in y_pred]          # add predicted label name
out["confidence"] = conf.round(4)                    # add confidence
out["correct"] = y_true == y_pred                    # add True/False correct
out[["text","label","pred","confidence","correct"]].to_csv(ROOT/"data"/"test_predictions.csv", index=False)  # save

# draw the confusion matrix as a picture (and save it as a png for the README)
fig, ax = plt.subplots(figsize=(5,4))                # make a blank chart
im = ax.imshow(cm, cmap="Blues")                     # color the grid by count
ax.set_xticks(range(3)); ax.set_xticklabels(LABELS, rotation=45, ha="right")  # x = predicted
ax.set_yticks(range(3)); ax.set_yticklabels(LABELS)  # y = true
ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title("Fine-tuned — Confusion Matrix")
for i in range(3):                                   # write the number in each box
    for j in range(3):
        ax.text(j, i, cm[i,j], ha="center", va="center", color="white" if cm[i,j]>cm.max()/2 else "black")
fig.colorbar(im); fig.tight_layout()                 # tidy up
fig.savefig(ROOT/"confusion_matrix.png", dpi=150)    # save the image
plt.show()                                           # show it here in the notebook

## 7. Groq zero-shot baseline
Ask `llama-3.3-70b-versatile` to label the **same 30 test comments** with no training, so we have something to compare against. Paste your Groq key when the box appears (it is NOT saved into the notebook).

In [ ]:
import os, re, time, getpass                         # os=env vars, re=text cleanup, time=pause, getpass=hidden input
from groq import Groq                                # the Groq API client

if not os.environ.get("GROQ_API_KEY"):               # if the key isn't already set...
    os.environ["GROQ_API_KEY"] = getpass.getpass("Groq API key: ")  # ...ask for it (hidden)
client = Groq(api_key=os.environ["GROQ_API_KEY"])    # connect to Groq

PROMPT = '''You are classifying r/soccer comments into exactly one label.

analysis = a structured argument backed by specific, verifiable evidence (stats, formations, named match events, history). The evidence is load-bearing.
hot_take = a bold, confident opinion with no evidence or only decorative/cherry-picked evidence. It asserts rather than argues.
reaction = an immediate emotional response to an event. Little to no argument, just a feeling.

Reply with ONLY one word: analysis, hot_take, or reaction. No punctuation, no explanation.

Comment: {text}'''                                   # the instructions we give the big model

def parse(r):                                         # clean the model's reply down to one label
    r = re.sub(r"[^a-z_ ]", "", r.strip().lower())   # lowercase, strip punctuation
    for lab in LABELS:                               # find which label it said
        if lab in r or lab.replace("_"," ") in r:
            return lab
    return None                                      # None if it said something weird

base_pred, unparsed = [], 0                          # collect predictions; count weird replies
for i, row in enumerate(test_df.itertuples(), 1):    # loop over each test comment
    resp = client.chat.completions.create(           # send it to Groq
        model="llama-3.3-70b-versatile",
        messages=[{"role":"user","content":PROMPT.format(text=row.text)}],
        temperature=0, max_tokens=5)                 # temp 0 = consistent; max 5 = one word
    lab = parse(resp.choices[0].message.content)     # read its answer
    if lab is None:                                  # if unparseable...
        unparsed += 1; lab = "hot_take"              # ...count it and fall back so the row still scores
    base_pred.append(lab)                            # store the prediction
    print(f"[{i}/{len(test_df)}] true={row.label:9s} pred={lab}")  # show progress live
    time.sleep(0.3)                                  # small pause = stay under free rate limit

base_acc = accuracy_score(test_df["label"], base_pred)  # baseline accuracy
base_report = classification_report(test_df["label"], base_pred, labels=LABELS, target_names=LABELS, output_dict=True, zero_division=0)  # per-class
print(f"\nBASELINE ACCURACY: {base_acc:.3f}  (unparsed: {unparsed})")  # show it
print(classification_report(test_df["label"], base_pred, labels=LABELS, target_names=LABELS, zero_division=0))

## 8. Compare + save results
Write `evaluation_results.json` (both models) used by the README.

In [ ]:
results = {                                          # build the results dictionary
  "test_set_size": int(len(test_df)),                # how many test comments
  "labels": LABELS,                                  # the label names
  "baseline": {                                      # ---- Groq baseline ----
      "model": "llama-3.3-70b-versatile", "approach": "zero-shot, temp 0",
      "accuracy": round(float(base_acc),4), "unparsed": unparsed,
      "per_class": {l:{k:round(base_report[l][k],4) for k in ("precision","recall","f1-score")} for l in LABELS},
      "macro_f1": round(base_report["macro avg"]["f1-score"],4)},
  "fine_tuned": {                                    # ---- fine-tuned DistilBERT ----
      "model": MODEL, "epochs": 10, "learning_rate": 5e-5, "batch_size": 16,
      "accuracy": round(float(ft_acc),4),
      "per_class": {l:{k:round(ft_report[l][k],4) for k in ("precision","recall","f1-score")} for l in LABELS},
      "macro_f1": round(ft_report["macro avg"]["f1-score"],4),
      "confusion_matrix": {"rows_true_cols_pred": LABELS, "matrix": cm.tolist()}},
}
(ROOT/"evaluation_results.json").write_text(json.dumps(results, indent=2))  # save to file
print("saved evaluation_results.json\n")             # confirm
print(f"BASELINE   accuracy: {base_acc:.3f}")        # compare the two models
print(f"FINE-TUNED accuracy: {ft_acc:.3f}")
print(f"improvement: {ft_acc-base_acc:+.3f}")        # did fine-tuning help?